In [4]:
import base64
from dotenv import load_dotenv
import os

from anthropic import Anthropic
from anthropic.types import Message

# Image Handling
    - Uses an image_block
    - Still need to use strong prompting techniques images
    - Up to 100 images across all messages in a single request
    - Max size of 5MB
    - max height/width of 8000px when sending a single image
    - Max height/width of 2000px when sending multiple images
    - Rough token calculation: 
      -  tokens = width px x height px / 750

In [5]:
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(api_key=anthropic_api_key)

model = "claude-sonnet-4-5-20250929"

In [6]:
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)

In [7]:
def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024    # max tokens for thinking when claude generates the response (1024 is min)
    ):
    params = {
        "model": model,
        "max_tokens": 4000,     # must be higher than 'thinking_budget'
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    if tools:
        params["tools"] = tools
    
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block for block in message.content if block.type == "text"]
    )

In [8]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [14]:
with open("../../data/images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8") 

messages = []

add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_011Cfrrv5g2RerXzEvs2nnpK', container=None, content=[TextBlock(citations=None, text='# Satellite Image Analysis - Fire Risk Assessment\n\n**1. Residence identification:** The primary residence is a light-colored, irregularly-shaped structure with multiple roof sections located in the center of the image, surrounded by dense forest vegetation on all sides with no visible driveway connection in this view.\n\n**2. Tree overhang analysis:** Multiple large trees with dense canopies are positioned immediately adjacent to and directly overhanging the residence, with approximately 50-75% of the roof area covered or closely shadowed by tree canopy from various directions.\n\n**3. Fire risk assessment:** The overhanging trees create multiple ember catch points across the majority of the roof surface, with dense canopy coverage providing continuous fuel paths directly to the structure and limited visibility of any defensible space clearance.\n\n**4. Defensible space identification: